# Multilingual Regional News Classifier (English / Hindi / Bengali)

A single multilingual model — built on LaBSE embeddings — classifies news headlines
into topics across three languages without any translation step. The notebook also
runs a zero-shot cross-lingual transfer test (train on English + Hindi, evaluate on
Bengali) to demonstrate that the shared embedding space generalizes across languages.

**Sections:**
1. Data loading
2. Category mapping & filtering
3. Preprocessing (cleaning + deduplication)
4. LaBSE embeddings
5. Sanity check: does the shared embedding space actually work?
6. Label audit
7. Baseline classifier + zero-shot diagnosis
8. Category merge & re-evaluation
9. Monolingual vs. multilingual comparison
10. Save artifacts for Streamlit deployment


In [ ]:
%cd /content/multilingual-regional-news-classifier

/content/multilingual-regional-news-classifier


## 1. Data loading

Pulling the L3Cube-IndicNews Short Headlines Classification (SHC) splits directly from GitHub.

In [1]:
import pandas as pd

urls = {
    ("bengali", "train"): "https://raw.githubusercontent.com/l3cube-pune/indic-nlp/main/L3Cube-IndicNews/Bengali/SHC/Bengal_SHC_Train.csv",
    ("bengali", "test"):  "https://raw.githubusercontent.com/l3cube-pune/indic-nlp/main/L3Cube-IndicNews/Bengali/SHC/Bengal_SHC_Test.csv",
    ("bengali", "valid"): "https://raw.githubusercontent.com/l3cube-pune/indic-nlp/main/L3Cube-IndicNews/Bengali/SHC/Bengal_SHC_Valid.csv",

    ("hindi", "train"): "https://raw.githubusercontent.com/l3cube-pune/indic-nlp/main/L3Cube-IndicNews/Hindi/SHC/Hindi_SHC_Train.csv",
    ("hindi", "test"):  "https://raw.githubusercontent.com/l3cube-pune/indic-nlp/main/L3Cube-IndicNews/Hindi/SHC/Hindi_SHC_Test.csv",
    ("hindi", "valid"): "https://raw.githubusercontent.com/l3cube-pune/indic-nlp/main/L3Cube-IndicNews/Hindi/SHC/Hindi_SHC_Valid.csv",

    ("english", "train"): "https://raw.githubusercontent.com/l3cube-pune/indic-nlp/main/L3Cube-IndicNews/English/SHC/English_train_shc.csv",
    ("english", "test"):  "https://raw.githubusercontent.com/l3cube-pune/indic-nlp/main/L3Cube-IndicNews/English/SHC/English_test_shc.csv",
    ("english", "valid"): "https://raw.githubusercontent.com/l3cube-pune/indic-nlp/main/L3Cube-IndicNews/English/SHC/English_valid_shc.csv",
}

dfs = []
for (lang, split), url in urls.items():
    d = pd.read_csv(url)
    d["language"] = lang
    d["split"] = split
    dfs.append(d)

combined = pd.concat(dfs, ignore_index=True)
print(combined.shape)
combined.head()


(114580, 4)


,labels,text,language,split
0,sports,শাহবাজ়ের লড়াকু ব্যাটিং! বাংলার প্রথম ইনিংস শ...,bengali,train
1,national,"বুলডোজার রাজ কাশ্মীরে, প্রতিবাদ",bengali,train
2,national,"হিংসা চলছেই, মণিপুরের গ্রামে খুন আরও ৩",bengali,train
3,international,সেন্ট পিটার্সবার্গে বিস্ফোরণ মৃত রুশ সামরিক ব...,bengali,train
4,kolkata,"যোধপুর পার্কে হঠাৎ উপড়ালো কৃষ্ণচূড়া, চাপা ...",bengali,train


## 2. Category mapping & filtering

Each language's raw labels differ in name and coverage. We map them onto a canonical
schema, then keep only the **6 categories with solid data in all three languages**
(politics, international, sports, entertainment, technology, national). Categories
present in only 1–2 languages (business, crime, health, education, lifestyle, auto,
editorial, regional) are dropped from the core model — documented as a known
limitation rather than force-fit.

In [2]:
category_map = {
    # Bengali
    "politics": "politics", "national": "national", "international": "international",
    "sports": "sports", "entertainment": "entertainment", "technology": "technology",
    "lifestyle": "lifestyle", "editorial": "editorial", "kolkata": "regional", "state": "regional",

    # Hindi
    "business": "business", "crime-news-hindi": "crime", "education": "education",
    "health-news-hindi": "health", "khel": "sports", "technology-news": "technology",
    "automobile": "auto",

    # English  (Note: "Science" is deliberately merged into "technology" here.)
    "Auto": "auto", "Business": "business", "Education": "education",
    "Elections": "politics", "Entertainment": "entertainment", "Health": "health",
    "India": "national", "Lifestyle": "lifestyle", "Science": "technology",
    "Sports": "sports", "Technology": "technology", "World": "international",
}

combined = combined.rename(columns={"labels": "raw_label"})
combined["raw_label"] = combined["raw_label"].str.strip()
combined = combined[combined["raw_label"] != ""]
combined["category"] = combined["raw_label"].map(category_map)
combined = combined.dropna(subset=["category", "text"])

print("Unmapped raw labels (should be empty):", combined[combined["category"].isna()]["raw_label"].unique())
print(combined.groupby(["language", "category"]).size().unstack(fill_value=0))


Unmapped raw labels (should be empty): []
category  auto  business  crime  editorial  education  entertainment  health  \
language                                                                       
bengali      0         0      0       2874          0           2925       0   
english   2146      4901      0          0       4594           3317    5017   
hindi     3379      3515   3455          0       3426           3600    3520   

category  international  lifestyle  national  politics  regional  sports  \
language                                                                   
bengali            2729       3099      3100      3100      5934    3100   
english            3635       4263      3636      4606         0    3376   
hindi              3507          0      3527      3510         0    3560   

category  technology  
language              
bengali         3100  
english         6579  
hindi           3523  


In [3]:
# Keeping only the 6 categories with solid coverage in all three languages
core_categories = ["politics", "international", "sports", "entertainment", "technology", "national"]
df = combined[combined["category"].isin(core_categories)][["text", "language", "category"]].reset_index(drop=True)
print(df.shape)
df.groupby(["language", "category"]).size().unstack(fill_value=0)


(64430, 3)


category,entertainment,international,national,politics,sports,technology
language,,,,,,
bengali,2925,2729,3100,3100,3100,3100
english,3317,3635,3636,4606,3376,6579
hindi,3600,3507,3527,3510,3560,3523


## 3. Preprocessing

**Principle: don't reuse English NLP habits on Hindi/Bengali.** No aggressive
lowercasing or stemming (script-specific and can corrupt Indic text). We apply
only script-agnostic cleaning: Unicode NFC normalization, whitespace collapsing,
and removal of zero-width joiner/non-joiner artifacts common in scraped Indic text.
Deduplication happens **before** embedding, since duplicate rows inflate category
counts and waste embedding compute.

In [4]:
import unicodedata
import re

def light_clean(text):
    text = unicodedata.normalize("NFC", str(text))
    text = re.sub(r"\s+", " ", text).strip()
    text = re.sub(r"[\u200b\u200c\u200d]", "", text)
    return text

df["text_clean"] = df["text"].apply(light_clean)
df = df[df["text_clean"].str.len() >= 3].reset_index(drop=True)

before = len(df)
df = df.drop_duplicates(subset="text_clean").reset_index(drop=True)
print(f"Removed {before - len(df)} duplicate rows, {len(df)} remaining")

df["char_len"] = df["text_clean"].str.len()
print(df.groupby("language")["char_len"].describe())


Removed 7565 duplicate rows, 56736 remaining
            count       mean        std   min   25%   50%    75%    max
language                                                               
bengali   17927.0  60.764712  19.805224  11.0  45.0  60.0   77.0  137.0
english   18993.0  90.508187  15.830779  13.0  80.0  91.0  101.0  177.0
hindi     19816.0  91.845125  29.631126  12.0  81.0  97.0  111.0  192.0


**Note on character length:** Bengali headlines show a lower mean character count
than Hindi/English. This is because of script density (Bengali conjuncts/matras carry more
information per Unicode character) and not shorter content. Raw character length isn't
comparable across scripts.

In [5]:
print(df.groupby(["language", "category"]).size().unstack(fill_value=0))
df.to_csv("preprocessed_news.csv", index=False)


category  entertainment  international  national  politics  sports  technology
language                                                                      
bengali            2919           2698      3099      3093    3019        3099
english            2789           3123      3061      2859    3163        3998
hindi              2409           3435      3486      3466    3510        3510


## 4. LaBSE embeddings

LaBSE maps all three languages into one shared 768-dim space — semantically similar
sentences land close together regardless of language, with no translation step.

In [6]:
!pip install -q sentence-transformers

from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("sentence-transformers/LaBSE")

# Confirm default max_seq_length (256) comfortably covers our headlines (token, not char, counts)
sample_lengths = [len(model.tokenizer.tokenize(t)) for t in df["text_clean"].sample(2000, random_state=42)]
print(pd.Series(sample_lengths).describe())


modules.json:   0%|          | 0.00/461 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/2.02k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/804 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/114 [00:00<?, ?B/s]

2_Dense/model.safetensors:   0%|          | 0.00/2.36M [00:00<?, ?B/s]

count    2000.00000
mean       21.22100
std         7.67024
min         3.00000
25%        16.00000
50%        21.00000
75%        26.00000
max        50.00000
dtype: float64


In [7]:
texts = df["text_clean"].tolist()

embeddings = model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

print(embeddings.shape)
np.save("labse_embeddings.npy", embeddings)
df.to_csv("preprocessed_news_final.csv", index=False)


Batches:   0%|          | 0/887 [00:00<?, ?it/s]

(56736, 768)


## 5. Sanity check: does the shared embedding space actually work?

Before training anything, verifying LaBSE's cross-lingual alignment directly: taking an
English headline, finding its nearest neighbors (by cosine similarity) in Bengali and
Hindi, and checking whether they're topically related with zero translation involved.

In [8]:
from sklearn.metrics.pairwise import cosine_similarity

def nearest_neighbors_check(anchor_indices, embeddings, df, other_langs=("bengali", "hindi"), top_k=3):
    for idx in anchor_indices:
        anchor_text = df.loc[idx, "text_clean"]
        anchor_cat = df.loc[idx, "category"]
        anchor_emb = embeddings[idx].reshape(1, -1)
        print(f"\n=== ANCHOR ({anchor_cat}): {anchor_text}")

        for lang in other_langs:
            lang_idxs = df[df["language"] == lang].index.to_numpy()
            lang_embs = embeddings[lang_idxs]
            sims = cosine_similarity(anchor_emb, lang_embs)[0]
            top = lang_idxs[np.argsort(sims)[::-1][:top_k]]

            print(f"  -- Top {top_k} nearest in {lang}:")
            for i in top:
                sim_val = sims[np.where(lang_idxs == i)[0][0]]
                print(f"     [{df.loc[i, 'category']}] ({sim_val:.3f}) {df.loc[i, 'text_clean']}")


anchor_idxs = df[df["language"] == "english"].groupby("category").head(1).index.tolist()
nearest_neighbors_check(anchor_idxs, embeddings, df)



=== ANCHOR (politics): Chhattisgarh Polls: Congress Releases First List, Fields CM Bhupesh Baghel From Patan
  -- Top 3 nearest in bengali:
     [national] (0.714) গুজরাট বিধানসভা নির্বাচন: প্রার্থীদের প্রথম তালিকা ঘোষণা বিজেপির
     [national] (0.634) Himachal Pradesh Election: হিমাচল প্রদেশ নির্বাচনে প্রার্থী তালিকা ঘোষণা গেরুয়া শিবিরের
     [national] (0.631) তিন রাজ্যের বিধানসভা ভোটের প্রথম তালিকা প্রকাশ কংগ্রেসের
  -- Top 3 nearest in hindi:
     [politics] (0.603) Kerala Exit poll: कांग्रेस को हराकर सत्ता में लौटेगा लेफ्ट, भाजपा पहली बार खोलेगी खाता
     [politics] (0.593) Assam Exit poll: पूर्वोत्तर में पहली बार कमल खिलने के आसार, 15 साल बाद कांग्रेस की होगी हार
     [national] (0.589) ‘छत्तीसगढ़ चुनाव में जीत मिली तो भूपेश बघेल के नाम पर सबसे पहले चर्चा होगी’, CM रेस से टीएस सिंह ने खुद को हटाया?

=== ANCHOR (national): TMC, Sadhvi Niranjan Jyoti Clash Over MGNREGA Dues, Political Drama In Bengal
  -- Top 3 nearest in bengali:
     [politics] (0.580) ত্রিপুরায় জোট রাজনীতিতে 

**Finding:** categories with clean, universal topics (technology, sports,
entertainment) retrieve strong same-topic matches across languages. Politics/national
anchors sometimes retrieve neighbors labeled the *other* of the two. This is not an embedding
failure, but a sign that "politics" vs. "national" is drawn inconsistently across
languages in the source data.

## 6. Label audit

Spot-checking a sample of the English `technology` category for label noise, since
the nearest-neighbor check above surfaced at least one clearly mislabeled row.

In [9]:
tech_english = df[(df["language"] == "english") & (df["category"] == "technology")]
print(f"Total technology rows (english): {len(tech_english)}")
for t in tech_english["text_clean"].sample(30, random_state=1):
    print("-", t)


Total technology rows (english): 3998
- After Rockstar Games, Insomniac Falls Victim To Massive Data Leak: Release Timeframe Of Wolverine, Venom Games Out Now
- Mysterious 'Black Widow' Binary With Shortest Orbit Yet Discovered. Here's What We Know So Far
- From Exit To Re-entry: A Quick Timeline Of Sam Altman's Sudden Ousting & Surprising Reinstatement As OpenAI CEO
- Top Tech News Today: Kaspersky Reveals Shopping Cyber Threats Ahead Of Black Friday, ChatGPT's Voice Chat Now Available For Free Users, More
- Amazon India Lays Off 500 Employees Across AWS, HR And Support Verticals: Report
- Strong Passwords, Regular Updates, More: How To Protect Your Personal & Professional Devices From Cyberattacks
- NASA Selects Axiom Space To Deliver Moonwalking Spacesuits For Artemis III Mission
- Tech-ing Care Of Cows: How Machine Learning, Blockchain Can Help Track Stray Cattle And Modernise Cow-Based Economy
- Ethics Of AI In India: Examining The Implications Of ChatGPT
- Garena Free Fire Max: E

**Finding:** after deduplication, this category is clean and dominated by genuine
tech/science headlines (AI, ISRO, smartphones, cybersecurity). Note that `Science`
was deliberately merged into `technology` in the mapping (Section 2); a handful of
unrelated headlines (politics/sports mislabeled as technology) remain, at a rate
consistent with normal scraped-news labeling noise, and are documented here rather
than hand-corrected.

## 7. Baseline classifier + zero-shot diagnosis

Two models, both LogisticRegression on top of frozen LaBSE embeddings:
- **Baseline**: trained on all three languages combined, evaluated per language.
- **Zero-shot**: trained on English + Hindi *only*, evaluated on Bengali (which the
  model never sees during training). This is the centerpiece cross-lingual transfer test.

In [10]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

X_train, X_test, y_train, y_test, lang_train, lang_test = train_test_split(
    embeddings, df["category"], df["language"],
    test_size=0.2, random_state=42, stratify=df["category"]
)

clf = LogisticRegression(max_iter=1000, class_weight="balanced")
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print("=== Baseline: all languages combined ===")
print(classification_report(y_test, y_pred))

for lang in ["english", "hindi", "bengali"]:
    mask = lang_test == lang
    print(f"\n=== {lang.upper()} only ===")
    print(classification_report(y_test[mask], y_pred[mask]))


=== Baseline: all languages combined ===
               precision    recall  f1-score   support

entertainment       0.83      0.89      0.86      1623
international       0.78      0.75      0.76      1851
     national       0.65      0.63      0.64      1929
     politics       0.76      0.76      0.76      1884
       sports       0.90      0.90      0.90      1939
   technology       0.91      0.91      0.91      2122

     accuracy                           0.81     11348
    macro avg       0.81      0.81      0.81     11348
 weighted avg       0.81      0.81      0.81     11348


=== ENGLISH only ===
               precision    recall  f1-score   support

entertainment       0.87      0.93      0.90       558
international       0.79      0.79      0.79       619
     national       0.69      0.70      0.70       591
     politics       0.88      0.86      0.87       569
       sports       0.96      0.92      0.94       656
   technology       0.89      0.89      0.89       78

In [11]:
# Zero-shot: train on English + Hindi, test on Bengali (unseen at train time)
train_mask = df["language"].isin(["english", "hindi"])
test_mask = df["language"] == "bengali"

X_train_zs = embeddings[train_mask.values]
y_train_zs = df.loc[train_mask, "category"]
X_test_zs = embeddings[test_mask.values]
y_test_zs = df.loc[test_mask, "category"]

clf_zs = LogisticRegression(max_iter=1000, class_weight="balanced")
clf_zs.fit(X_train_zs, y_train_zs)
y_pred_zs = clf_zs.predict(X_test_zs)

print("=== ZERO-SHOT: trained on English+Hindi, tested on Bengali ===")
print(classification_report(y_test_zs, y_pred_zs))


=== ZERO-SHOT: trained on English+Hindi, tested on Bengali ===
               precision    recall  f1-score   support

entertainment       0.82      0.75      0.78      2919
international       0.61      0.72      0.66      2698
     national       0.35      0.45      0.39      3099
     politics       0.37      0.38      0.37      3093
       sports       0.92      0.66      0.77      3019
   technology       0.95      0.82      0.88      3099

     accuracy                           0.63     17927
    macro avg       0.67      0.63      0.64     17927
 weighted avg       0.67      0.63      0.64     17927



In [12]:
# Diagnose where zero-shot error concentrates
labels = sorted(y_test_zs.unique())
cm = confusion_matrix(y_test_zs, y_pred_zs, labels=labels)
cm_df = pd.DataFrame(cm, index=labels, columns=labels)
cm_norm = cm_df.div(cm_df.sum(axis=1), axis=0).round(2)

print("Row-normalized confusion matrix (proportion of true label predicted as each class):")
cm_norm


Row-normalized confusion matrix (proportion of true label predicted as each class):


,entertainment,international,national,politics,sports,technology
entertainment,0.75,0.04,0.12,0.06,0.01,0.01
international,0.02,0.72,0.10,0.14,0.01,0.01
national,0.03,0.15,0.45,0.34,0.02,0.02
politics,0.03,0.05,0.53,0.38,0.01,0.00
sports,0.06,0.12,0.05,0.11,0.66,0.01
technology,0.02,0.05,0.06,0.03,0.01,0.82


**Diagnosis:** error concentrates almost entirely between `politics` and
`national` (in both directions) which is confirmed by the confusion matrix above, and
consistent with the ambiguity spotted in Section 5's sanity check. Every other
category is comparatively clean. This is a labeling-schema issue, not a model
or embedding weakness. So the fix is to merge the two categories and not to tune
the model further.

## 8. Category merge & re-evaluation

Merging `politics` + `national` into a single `politics_national` category and
re-running both the baseline and zero-shot evaluations.

In [13]:
merge_map = {"national": "politics_national", "politics": "politics_national"}
df["category_merged"] = df["category"].replace(merge_map)

X_train, X_test, y_train, y_test, lang_train, lang_test = train_test_split(
    embeddings, df["category_merged"], df["language"],
    test_size=0.2, random_state=42, stratify=df["category_merged"]
)

clf = LogisticRegression(max_iter=1000, class_weight="balanced")
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print("=== Baseline (merged categories) ===")
print(classification_report(y_test, y_pred))

for lang in ["english", "hindi", "bengali"]:
    mask = lang_test == lang
    print(f"\n=== {lang.upper()} only (merged) ===")
    print(classification_report(y_test[mask], y_pred[mask]))


=== Baseline (merged categories) ===
                   precision    recall  f1-score   support

    entertainment       0.81      0.90      0.85      1623
    international       0.74      0.77      0.76      1851
politics_national       0.90      0.81      0.85      3813
           sports       0.88      0.90      0.89      1939
       technology       0.89      0.92      0.90      2122

         accuracy                           0.85     11348
        macro avg       0.84      0.86      0.85     11348
     weighted avg       0.86      0.85      0.85     11348


=== ENGLISH only (merged) ===
                   precision    recall  f1-score   support

    entertainment       0.86      0.94      0.90       558
    international       0.75      0.82      0.78       619
politics_national       0.92      0.84      0.88      1199
           sports       0.94      0.91      0.93       655
       technology       0.87      0.89      0.88       788

         accuracy                         

In [14]:
# Zero-shot, merged categories
train_mask = df["language"].isin(["english", "hindi"])
test_mask = df["language"] == "bengali"

X_train_zs = embeddings[train_mask.values]
y_train_zs = df.loc[train_mask, "category_merged"]
X_test_zs = embeddings[test_mask.values]
y_test_zs = df.loc[test_mask, "category_merged"]

clf_zs = LogisticRegression(max_iter=1000, class_weight="balanced")
clf_zs.fit(X_train_zs, y_train_zs)
y_pred_zs = clf_zs.predict(X_test_zs)

print("=== ZERO-SHOT (merged categories): English+Hindi -> Bengali ===")
print(classification_report(y_test_zs, y_pred_zs))


=== ZERO-SHOT (merged categories): English+Hindi -> Bengali ===
                   precision    recall  f1-score   support

    entertainment       0.81      0.77      0.79      2919
    international       0.58      0.71      0.64      2698
politics_national       0.74      0.83      0.78      6192
           sports       0.91      0.67      0.77      3019
       technology       0.94      0.83      0.88      3099

         accuracy                           0.77     17927
        macro avg       0.80      0.76      0.77     17927
     weighted avg       0.79      0.77      0.78     17927



**Result:** zero-shot Bengali accuracy improves from **63% → 77%** after the
merge, and `politics_national`'s zero-shot F1 goes from ~0.37 to **0.78** —
confirming the diagnosis was correct. Remaining gaps (e.g. `sports` recall,
`international` F1) reflect genuine cross-lingual transfer difficulty rather
than a labeling artifact.

## 9. Monolingual vs. multilingual comparison

Does a single shared multilingual model actually beat three separate
per-language models? Train one LogisticRegression per language (own data only,
merged categories) and compare.

In [15]:
from sklearn.metrics import accuracy_score, f1_score

monolingual_results = {}

for lang in ["english", "hindi", "bengali"]:
    lang_mask = (df["language"] == lang).values
    X_lang = embeddings[lang_mask]
    y_lang = df.loc[lang_mask, "category_merged"]

    X_tr, X_te, y_tr, y_te = train_test_split(
        X_lang, y_lang, test_size=0.2, random_state=42, stratify=y_lang
    )

    mono_clf = LogisticRegression(max_iter=1000, class_weight="balanced")
    mono_clf.fit(X_tr, y_tr)
    y_pred_mono = mono_clf.predict(X_te)

    monolingual_results[lang] = {
        "accuracy": accuracy_score(y_te, y_pred_mono),
        "macro_f1": f1_score(y_te, y_pred_mono, average="macro"),
    }

    labels = sorted(y_te.unique())
    cm_df = pd.DataFrame(confusion_matrix(y_te, y_pred_mono, labels=labels), index=labels, columns=labels)
    cm_norm = cm_df.div(cm_df.sum(axis=1), axis=0).round(2)

    print(f"=== Monolingual {lang.upper()} ===")
    print(classification_report(y_te, y_pred_mono))
    print(f"Confusion matrix (row-normalized):\n{cm_norm}\n")

print(monolingual_results)


=== Monolingual ENGLISH ===
                   precision    recall  f1-score   support

    entertainment       0.90      0.95      0.93       558
    international       0.76      0.85      0.80       624
politics_national       0.93      0.86      0.90      1184
           sports       0.95      0.93      0.94       633
       technology       0.90      0.90      0.90       800

         accuracy                           0.89      3799
        macro avg       0.89      0.90      0.89      3799
     weighted avg       0.90      0.89      0.89      3799

Confusion matrix (row-normalized):
                   entertainment  international  politics_national  sports  \
entertainment               0.95           0.01               0.01    0.02   
international               0.01           0.85               0.06    0.00   
politics_national           0.02           0.09               0.86    0.01   
sports                      0.03           0.01               0.02    0.93   
technology   

In [16]:
# Multilingual accuracy figures are taken from Section 8's per-language "merged" results
multilingual_accuracy = {"english": 0.87, "hindi": 0.86, "bengali": 0.83}  # update if Section 8 numbers change

comparison = pd.DataFrame({
    "Language": ["English", "Hindi", "Bengali"],
    "Monolingual Accuracy": [monolingual_results[l]["accuracy"] for l in ["english", "hindi", "bengali"]],
    "Multilingual Accuracy": [multilingual_accuracy[l] for l in ["english", "hindi", "bengali"]],
})
comparison["Difference"] = comparison["Multilingual Accuracy"] - comparison["Monolingual Accuracy"]
comparison


,Language,Monolingual Accuracy,Multilingual Accuracy,Difference
0,English,0.892340,0.87,-0.022340
1,Hindi,0.878658,0.86,-0.018658
2,Bengali,0.853318,0.83,-0.023318


**Result:** monolingual models edge out the multilingual model by ~2 points per
language — a real, well-documented capacity-sharing tradeoff, not a bug. But the
multilingual model does something no monolingual model can: classify a language
with **zero labeled training data** (77% zero-shot accuracy on Bengali). The
per-language confusion matrices above also show `international` is notably
cleaner monolingually (0.76–0.85 diagonal) than in the multilingual zero-shot
case (0.64 F1) — the accuracy cost of sharing one embedding space concentrates in
categories with cross-lingual reporting-style differences, while universal
categories (technology, sports) lose almost nothing either way.

## 10. Phase 1: sampled artifacts for initial deployment

For a fast first deployment, save a sampled reference set (150 rows per
language/category) rather than the full dataset — small enough to commit
directly to the GitHub repo alongside the app, no external hosting needed.

**Superseded by Section 11** once the full dataset was moved to Hugging Face
Hub for richer nearest-neighbor results — kept here to document the
two-phase deployment approach.

In [17]:
import joblib

joblib.dump(clf, "multilingual_clf.joblib")   # trained on all 3 languages, merged categories (Section 8)
joblib.dump(clf_zs, "zeroshot_clf.joblib")    # trained on English+Hindi only (Section 8)

ref_sample = (
    df.groupby(["language", "category_merged"], group_keys=False)
      .apply(lambda x: x.sample(min(len(x), 150), random_state=42))
      .reset_index(drop=True)
)
ref_embeddings = embeddings[df.index.isin(ref_sample.index)]

ref_sample.to_csv("reference_data.csv", index=False)
np.save("reference_embeddings.npy", ref_embeddings)

print(f"Saved {len(ref_sample)} reference rows for nearest-neighbor lookup")
print(f"Multilingual classes: {clf.classes_}")
print(f"Zero-shot classes: {clf_zs.classes_}")


Saved 2250 reference rows for nearest-neighbor lookup
Multilingual classes: ['entertainment' 'international' 'politics_national' 'sports' 'technology']
Zero-shot classes: ['entertainment' 'international' 'politics_national' 'sports' 'technology']


/tmp/ipykernel_969/1853234781.py:8: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(min(len(x), 150), random_state=42))


## 11. Phase 2: full-dataset artifacts for deployment

The initial deployment used a sampled reference set (150 rows per language/category)
for the nearest-neighbor feature. This step saves the full deduplicated dataset
(~56,700 rows) and its embeddings instead, hosted on Hugging Face Hub and pulled
by the Streamlit app at runtime via `huggingface_hub.hf_hub_download`.

In [19]:
!pip install -q pyarrow
import numpy as np

df.to_parquet("full_reference_data.parquet", index=False)
np.save("full_reference_embeddings.npy", embeddings)

print(df.shape, embeddings.shape)

(56736, 6) (56736, 768)


In [20]:
check_df = pd.read_parquet("full_reference_data.parquet")
print(check_df.columns.tolist())
print(check_df.shape)

['text', 'language', 'category', 'text_clean', 'char_len', 'category_merged']
(56736, 6)
